In [ ]:
# !pip install librosa soundfile pillow scikit-learn -q

In [ ]:
# DATA & UTILIDADES

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import librosa
from sklearn.model_selection import train_test_split


In [ ]:
pd.set_option('display.float_format', lambda x: '%.1f' % x)

import warnings

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ruta Drive:
base_path = "/content/drive/MyDrive/WAV_AL_MELS"

# (Opcional) Verificar que exista la ruta
import os
print("¿Existe la ruta?", os.path.exists(base_path))
print("Contenido en base_path:", os.listdir(base_path)[:10])  # muestra primeras carpetas/archivos

Mounted at /content/drive
¿Existe la ruta? True
Contenido en base_path: ['BI', 'SI', 'GE', 'TM', 'MH', 'VM']


In [ ]:
classes = []
counts = []

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        n_wav = len([f for f in os.listdir(folder_path) if f.lower().endswith(".wav")])
        classes.append(folder)
        counts.append(n_wav)

df_counts = pd.DataFrame({"Clase": classes, "Cantidad_wav": counts}).sort_values("Cantidad_wav", ascending=False)
df_counts

,Clase,Cantidad_wav
4,TM,3078
5,VM,1956
2,MH,1122
3,SI,337
0,BI,326
1,GE,319


In [ ]:
AUDIO_ROOT = Path("/content/drive/MyDrive/WAV_AL_MELS")   # <- cambia esta ruta
SPEC_ROOT = Path("/content/drive/MyDrive/spec_dataset")   # <- cambia esta ruta

SPEC_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# PARÁMETROS
SR = 22050
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
IMG_EXT = ".png"

TRAIN_SIZE = 0.70
VAL_SIZE   = 0.15
TEST_SIZE  = 0.15

if round(TRAIN_SIZE + VAL_SIZE + TEST_SIZE, 5) != 1.0:
    raise ValueError("TRAIN_SIZE + VAL_SIZE + TEST_SIZE deben sumar 1.0")

In [ ]:
CLASSES = sorted([p.name for p in AUDIO_ROOT.iterdir() if p.is_dir()])

if not CLASSES:
    raise ValueError(f"No se encontraron carpetas de clases dentro de: {AUDIO_ROOT}")

print("Clases detectadas:")
for c in CLASSES:
    print(f" - {c}")

Clases detectadas:
 - BI
 - GE
 - MH
 - SI
 - TM
 - VM


In [ ]:
DURATION = 2
TARGET_SAMPLES = SR * DURATION

def split_files_by_class(file_list, train_size=TRAIN_SIZE, val_size=VAL_SIZE, test_size=TEST_SIZE):
    """
    Divide archivos (no segmentos) en train/val/test.
    Esto evita que ventanas del mismo audio queden en distintos conjuntos.
    """
    total = len(file_list)

    if total == 0:
        return [], [], []

    if total == 1:
        return file_list, [], []

    if total == 2:
        return [file_list[0]], [file_list[1]], []

    temp_size = val_size + test_size

    train_files, temp_files = train_test_split(
        file_list,
        test_size=temp_size,
        random_state=42,
        shuffle=True
    )

    if len(temp_files) == 0:
        return train_files, [], []

    if len(temp_files) == 1:
        return train_files, temp_files, []

    test_ratio_within_temp = test_size / (val_size + test_size)

    val_files, test_files = train_test_split(
        temp_files,
        test_size=test_ratio_within_temp,
        random_state=42,
        shuffle=True
    )

    return train_files, val_files, test_files


def segment_audio(y, target_samples=TARGET_SAMPLES):
    """
    Parte una señal completa en ventanas consecutivas de 5 segundos.
    Si el último segmento queda corto, lo rellena con ceros.
    """
    total_samples = len(y)

    if total_samples == 0:
        return []

    n_segments = int(np.ceil(total_samples / target_samples))
    segments = []

    for i in range(n_segments):
        start = i * target_samples
        end = min(start + target_samples, total_samples)

        seg = y[start:end]

        if len(seg) < target_samples:
            seg = np.pad(seg, (0, target_samples - len(seg)))

        segments.append(seg)

    return segments


def save_segment_as_spectrogram(y_segment, output_path,
                                sr=SR,
                                n_fft=N_FFT,
                                hop_length=HOP_LENGTH,
                                n_mels=N_MELS):
    """
    Convierte un segmento de audio ya recortado (5 s) en un mel-espectrograma
    grayscale normalizado y lo guarda en PNG.
    """
    mel_spec = librosa.feature.melspectrogram(
        y=y_segment,
        sr=sr,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        power=2.0
    )

    mel_db = librosa.power_to_db(mel_spec, ref=np.max)

    mel_min = mel_db.min()
    mel_max = mel_db.max()

    if mel_max - mel_min == 0:
        mel_norm = np.zeros_like(mel_db, dtype=np.float32)
    else:
        mel_norm = (mel_db - mel_min) / (mel_max - mel_min)

    img_array = (mel_norm * 255).astype(np.uint8)
    img_array = np.flipud(img_array)  # opcional, visualmente más natural

    img = Image.fromarray(img_array, mode="L")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    img.save(output_path)


In [ ]:
resumen = []
detalle_archivos = []
errores = []

for cls in CLASSES:
    class_dir = AUDIO_ROOT / cls
    wav_files = sorted(list(class_dir.glob("*.wav")))

    if len(wav_files) == 0:
        print(f"[AVISO] La clase '{cls}' no tiene archivos .wav")
        continue

    train_files, val_files, test_files = split_files_by_class(wav_files)

    split_map = {
        "train": train_files,
        "val": val_files,
        "test": test_files
    }

    for split_name, files_in_split in split_map.items():
        split_output_dir = SPEC_ROOT / split_name / cls

        total_audios_split = 0
        total_specs_split = 0

        for wav_path in files_in_split:
            try:
                # Cargar audio completo a 22050 Hz y mono
                y, _ = librosa.load(wav_path, sr=SR, mono=True)

                if y is None or len(y) == 0:
                    raise ValueError("Audio vacío o inválido")

                segments = segment_audio(y, target_samples=TARGET_SAMPLES)

                specs_generados_archivo = 0

                for idx, seg in enumerate(segments, start=1):
                    output_name = f"{wav_path.stem}_seg_{idx:04d}{IMG_EXT}"
                    output_path = split_output_dir / output_name

                    save_segment_as_spectrogram(seg, output_path)
                    specs_generados_archivo += 1

                total_audios_split += 1
                total_specs_split += specs_generados_archivo

                detalle_archivos.append({
                    "clase": cls,
                    "split": split_name,
                    "audio_fuente": str(wav_path),
                    "segmentos_5s_generados": specs_generados_archivo,
                    "ruta_destino": str(split_output_dir)
                })

            except Exception as e:
                errores.append({
                    "clase": cls,
                    "split": split_name,
                    "archivo": str(wav_path),
                    "error": str(e)
                })

        resumen.append({
            "split": split_name,
            "clase": cls,
            "ruta": str(split_output_dir),
            "audios_fuente": total_audios_split,
            "espectrogramas_5s_generados": total_specs_split
        })

print("Proceso terminado.")

Proceso terminado.


In [ ]:
df_resumen = pd.DataFrame(resumen)

if df_resumen.empty:
    print("No se generaron espectrogramas.")
else:
    df_resumen = df_resumen.sort_values(["clase", "split"]).reset_index(drop=True)

    print("Resumen por clase y split:")
    display(df_resumen)

    df_totales_clase = (
        df_resumen.groupby("clase", as_index=False)[["audios_fuente", "espectrogramas_5s_generados"]]
        .sum()
        .rename(columns={
            "audios_fuente": "total_audios_fuente",
            "espectrogramas_5s_generados": "total_espectrogramas_5s"
        })
    )

    print("Totales por clase:")
    display(df_totales_clase)

Resumen por clase y split:


,split,clase,ruta,audios_fuente,espectrogramas_5s_generados
0,test,BI,/content/drive/MyDrive/spec_dataset/test/BI,49,49
1,train,BI,/content/drive/MyDrive/spec_dataset/train/BI,228,228
2,val,BI,/content/drive/MyDrive/spec_dataset/val/BI,49,49
3,test,GE,/content/drive/MyDrive/spec_dataset/test/GE,48,48
4,train,GE,/content/drive/MyDrive/spec_dataset/train/GE,223,223
5,val,GE,/content/drive/MyDrive/spec_dataset/val/GE,48,48
6,test,MH,/content/drive/MyDrive/spec_dataset/test/MH,169,169
7,train,MH,/content/drive/MyDrive/spec_dataset/train/MH,785,785
8,val,MH,/content/drive/MyDrive/spec_dataset/val/MH,168,168
9,test,SI,/content/drive/MyDrive/spec_dataset/test/SI,51,51


Totales por clase:


,clase,total_audios_fuente,total_espectrogramas_5s
0,BI,326,326
1,GE,319,319
2,MH,1122,1122
3,SI,337,337
4,TM,3078,3078
5,VM,1956,1956


In [ ]:
df_detalle = pd.DataFrame(detalle_archivos)

if df_detalle.empty:
    print("No hay detalle por archivo.")
else:
    print("Detalle por archivo fuente:")
    display(df_detalle.head(20))

Detalle por archivo fuente:


,clase,split,audio_fuente,segmentos_5s_generados,ruta_destino
0,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_25_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
1,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/VNE_101_...,1,/content/drive/MyDrive/spec_dataset/train/BI
2,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_22_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
3,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/SQC_62_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
4,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_06_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
5,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/VNE_14_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
6,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_23_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
7,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_35_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
8,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/SQC_56_P...,1,/content/drive/MyDrive/spec_dataset/train/BI
9,BI,train,/content/drive/MyDrive/WAV_AL_MELS/BI/PPN_44_P...,1,/content/drive/MyDrive/spec_dataset/train/BI


In [ ]:
if not df_resumen.empty:
    resumen_path = SPEC_ROOT / "resumen_por_clase_y_split.csv"
    totales_path = SPEC_ROOT / "totales_por_clase.csv"

    df_resumen.to_csv(resumen_path, index=False, encoding="utf-8-sig")
    df_totales_clase.to_csv(totales_path, index=False, encoding="utf-8-sig")

    print(f"Resumen guardado en: {resumen_path}")
    print(f"Totales guardados en: {totales_path}")

if not df_detalle.empty:
    detalle_path = SPEC_ROOT / "detalle_por_audio_fuente.csv"
    df_detalle.to_csv(detalle_path, index=False, encoding="utf-8-sig")
    print(f"Detalle guardado en: {detalle_path}")

if errores:
    df_errores = pd.DataFrame(errores)
    errores_path = SPEC_ROOT / "errores_conversion.csv"
    df_errores.to_csv(errores_path, index=False, encoding="utf-8-sig")

    print(f"Errores guardados en: {errores_path}")
    display(df_errores.head())
else:
    print("Proceso finalizado sin errores.")

Resumen guardado en: /content/drive/MyDrive/spec_dataset/resumen_por_clase_y_split.csv
Totales guardados en: /content/drive/MyDrive/spec_dataset/totales_por_clase.csv
Detalle guardado en: /content/drive/MyDrive/spec_dataset/detalle_por_audio_fuente.csv
Proceso finalizado sin errores.
